## this notebook checks;
### 1. augmenting with previously collected data that is not in the new
### 2. how many  reviews we have classified from the last itereation haw many are left

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

## All pools

In [ ]:
## list used  in the pool selection process
list_place_types = ['aquatic center', 'indoor swimming pool', 'lake', 'lake shore swimming area','lido',
                    'outdoor bath', 'outdoor swimming pool', 'public bath', 'public bathroom', 
                    'public sauna', 'public beach','public swimming pool', 'recreation center', 'spa','spa and health club',
                    'sports complex', 'swimming facility', 'swimming lake','swimming pool', 'thermal baths',
                    'water', 'water park']

In [ ]:
swimming_pools = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_with_reviews.parquet")
swimming_pools.info()

In [ ]:
#are the missing pools checking withthe state capitals?
previous_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/final_swimmingpool_list_previous.csv")
previous_pools .info()

In [ ]:
state_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/state_capital_pools.csv")
state_pools.info()

In [ ]:
missed_previous_pools = pd.concat([
    previous_pools[
        (~previous_pools['unique_identifier'].isin(swimming_pools['unique_identifier'])) |
        (~previous_pools['place_id'].isin(swimming_pools['place_id']))
    ],
    state_pools[
        ~state_pools['place_id'].isin(swimming_pools['place_id'])
    ].reset_index(drop=True)
]).reset_index(drop=True)
#put columns in lower case
missed_previous_pools["primary_place_type"] = missed_previous_pools["primary_place_type"].str.lower()
missed_previous_pools.info()

In [ ]:
missed_previous_pools.primary_place_type.unique()

### Somehow there are  swimming pools from the previous iteration that are not in the new one :(
### we would have  to merge them

In [ ]:
merge_cols = swimming_pools.columns.intersection(missed_previous_pools.columns).tolist()
final_list_pools = pd.concat([swimming_pools[merge_cols], missed_previous_pools[merge_cols]], ignore_index=True)
#put columns in lower case
final_list_pools['primary_place_type'] = final_list_pools['primary_place_type'].str.lower()
final_list_pools.info()

In [ ]:
final_list_pools[~final_list_pools['primary_place_type'].isin(list_place_types)]

In [ ]:
## thas
final_list_pools.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_updated_13042026.parquet", index=False)


In [ ]:
actual_pools_with_reviews = final_list_pools[final_list_pools['total_reviews'].notna()].reset_index(drop=True)
actual_pools_with_reviews.info()

In [ ]:
actual_pools_with_reviews.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_with_reviews_updated_13042026.parquet", index=False)

## All reviews

In [ ]:
all_reviews = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/clean_datasets/merged_reviews.parquet")
all_reviews.info()

In [ ]:
#newly collected reviews as of 30.04.2026
new_reviews = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_reviews.parquet")
new_reviews.info()

In [ ]:
previous_reviews = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/ICWSM_15012026/working_data/all_collected_reviews_25112025.parquet")
previous_reviews.info()

In [ ]:
state_pools_reviews = pd.read_csv(
    "/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/state_capital_pools_reviews.csv",
    dtype={"reviewer_id": "string", "uploaded_pics": "string"},
)
state_pools_reviews.info()

In [ ]:
missed_previous_pools_reviews = pd.concat([
    previous_reviews[
        (~previous_reviews['review_id'].isin(all_reviews['review_id']))
    ],
    state_pools_reviews[
        ~state_pools_reviews['review_id'].isin(all_reviews['review_id'])
    ].reset_index(drop=True)
]).reset_index(drop=True)
#put columns in lower case
missed_previous_pools_reviews["primary_place_type"] = missed_previous_pools_reviews["primary_place_type"].str.lower()
missed_previous_pools_reviews.info()

## Somehow there are  swimming pools reviews from the previous iteration that are not in the new one :(
### we would have  to merge them

In [ ]:
all_reviews_updated = pd.concat([all_reviews,new_reviews,missed_previous_pools_reviews], ignore_index=True).drop_duplicates(subset=['review_id'], keep='last').reset_index(drop=True)
all_reviews_updated.info()

In [ ]:
#save to processed data
all_reviews_updated.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/final_all_reviews_updated_30042026.parquet")

In [ ]:
# check for missed reviews for the actual pools with reviews
actual_reviews_missing = (
    actual_pools_with_reviews.groupby("unique_identifier", as_index=False)
    .agg(
        place_name=("place_name", "first"),
        total_reviews=("total_reviews", "first"),
    )
    .merge(
        all_reviews_updated.groupby("unique_identifier", as_index=False)["review_id"].count().rename(
            columns={"review_id": "collected_reviews"}
        ),
        on="unique_identifier",
        how="left",
    )
)

actual_reviews_missing["collected_reviews"] = actual_reviews_missing["collected_reviews"].fillna(0).astype("int64")
actual_reviews_missing["total_reviews"] = actual_reviews_missing["total_reviews"].fillna(0).astype("int64")
actual_reviews_missing["missing_reviews"] = (
    actual_reviews_missing["total_reviews"] - actual_reviews_missing["collected_reviews"]
)

print("Total missing reviews:", int(actual_reviews_missing["missing_reviews"].clip(lower=0).sum()))
actual_reviews_missing[actual_reviews_missing["missing_reviews"] > 0].sort_values("missing_reviews", ascending=False)[
    ["place_name", "unique_identifier", "total_reviews", "collected_reviews", "missing_reviews"]
]

In [ ]:
# check for missed reviews for the actual pools with reviews
actual_reviews_missing = actual_pools_with_reviews.copy()
actual_reviews_missing["review_match_id"] = actual_reviews_missing["unique_identifier"].fillna(actual_reviews_missing["place_id"])

all_reviews_updated_with_match_id = all_reviews_updated.copy()
all_reviews_updated_with_match_id["review_match_id"] = all_reviews_updated_with_match_id["unique_identifier"].fillna(
    all_reviews_updated_with_match_id["place_id"]
)

actual_reviews_missing = (
    actual_reviews_missing.groupby("review_match_id", as_index=False)
    .agg(
        place_name=("place_name", "first"),
        unique_identifier=("unique_identifier", "first"),
        place_id=("place_id", "first"),
        total_reviews=("total_reviews", "first"),
    )
    .merge(
        all_reviews_updated_with_match_id.groupby("review_match_id", as_index=False)["review_id"].count().rename(
            columns={"review_id": "collected_reviews"}
        ),
        on="review_match_id",
        how="left",
    )
)

actual_reviews_missing["collected_reviews"] = actual_reviews_missing["collected_reviews"].fillna(0).astype("int64")
actual_reviews_missing["total_reviews"] = actual_reviews_missing["total_reviews"].fillna(0).astype("int64")
actual_reviews_missing["missing_reviews"] = (
    actual_reviews_missing["total_reviews"] - actual_reviews_missing["collected_reviews"]
).clip(lower=0)

# Extract the latest next_key by retrieved_time for each place
next_key_by_place = (
    all_reviews_updated_with_match_id.sort_values("retrieved_time", ascending=False)
    .groupby("review_match_id", as_index=False)
    .agg(next_key=("next_key", "first"))
)

# Merge next_key into the missing reviews summary
actual_reviews_missing = actual_reviews_missing.merge(
    next_key_by_place,
    on="review_match_id",
    how="left"
)

# Calculate expected_jsons and collected_jsons columns
actual_reviews_missing["expected_jsons"] = np.ceil(actual_reviews_missing["total_reviews"] / 10).astype("int64")
actual_reviews_missing["collected_jsons"] = np.where(
    actual_reviews_missing["collected_reviews"] == 0,
    0,
    np.where(
        actual_reviews_missing["collected_reviews"] < 10,
        1,
        np.ceil(actual_reviews_missing["collected_reviews"] / 10)
    )
).astype("int64")

print("Total missing reviews:", int(actual_reviews_missing["missing_reviews"].sum()))
actual_reviews_missing.sort_values("missing_reviews", ascending=False)[
    ["place_name", "unique_identifier", "place_id", "total_reviews", "collected_reviews", "missing_reviews", "expected_jsons", "collected_jsons", "next_key"]
]

In [ ]:
actual_reviews_missing.next_key.nunique()

In [ ]:
actual_reviews_missing[['place_id', 'unique_identifier', 'expected_jsons', 'collected_jsons', 'next_key']].to_csv("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/missing_reviews_16042026.csv", index=False)

In [ ]:
#now select the reviews for the actual pools 
actual_pools_reviews = all_reviews_updated[(all_reviews_updated['place_id'].isin(final_list_pools['place_id']))| (all_reviews_updated['unique_identifier'].isin(final_list_pools['unique_identifier']))].reset_index(drop=True)
actual_pools_reviews.info()

In [ ]:
#check for missed reviews 

In [ ]:
actual_pools_reviews.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_reviews_updated_13042026.parquet", index=False)

##  select text reviews

In [ ]:
text_swim_reviews = actual_pools_reviews[actual_pools_reviews["org_review"].notnull() & (actual_pools_reviews["org_review"].str.strip() != "")].reset_index(drop=True)
text_swim_reviews.info()

In [ ]:
text_swim_reviews.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_text_reviews_13042026.parquet", index=False)

## Which of this were already summarized and classified

In [ ]:
previously_classified = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/ICWSM_15012026/working_data/merged_700k_classified_with_ethnicity_14012026.parquet")
previously_classified.info()

In [ ]:
previously_classified[~previously_classified['review_id'].isin(final_all_reviews['review_id'])].info()

In [ ]:
import pandas as pd
racial_reviews =pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/ICWSM_15012026/working_data/final_racialreviews_from_700k_15012026.parquet')
racial_reviews.info()

In [ ]:
racial_reviews.columns

In [ ]:
racial_reviews[['review_id', 'place_id',  'org_review', 'trans_review']].to_csv('racial_reviews.csv', index=False)

In [ ]:
# Length (number of characters) for original and translated review text
review_lengths = racial_reviews[['review_id', 'place_id', 'org_review', 'trans_review']].copy()
review_lengths['org_review_len'] = review_lengths['org_review'].fillna('').str.len()
review_lengths['trans_review_len'] = review_lengths['trans_review'].fillna('').str.len()

# Preview
display(review_lengths[['review_id', 'place_id', 'org_review_len', 'trans_review_len']].head(5))

# Summary statistics
display(review_lengths[['org_review_len', 'trans_review_len']].describe())